# Tableau Web Scraper with Bootstrap Session

This notebook captures a Tableau bootstrap session URL and uses it to scrape data from the Tableau dashboard.
It demonstrates the integration between the bootstrap session capture process and the scraper4 module.

In [1]:
import sys
import os
import json
import asyncio
from pathlib import Path

# Add the Attempt1 directory to the Python path
attempt1_dir = Path(".").resolve()
if str(attempt1_dir) not in sys.path:
    sys.path.insert(0, str(attempt1_dir))

print(f"Added to path: {attempt1_dir}")
print(f"Current working directory: {os.getcwd()}")
print(f"Python path: {sys.path[:3]}...")  # Show first 3 entries

Added to path: C:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt1
Current working directory: c:\Users\20203525\Documents\2025 2026\WB4U\wb4y-webscraper\Attempt1
Python path: ['C:\\Users\\20203525\\Documents\\2025 2026\\WB4U\\wb4y-webscraper\\Attempt1', 'c:\\Users\\20203525\\Documents\\2025 2026\\WB4U\\wb4y-webscraper\\Attempt1', 'c:\\Users\\20203525\\AppData\\Local\\anaconda3\\python312.zip']...


## Section 1: Import Required Libraries

Import necessary libraries for bootstrap session capture and web scraping operations.

In [2]:
# Import the scripts
from captureBootstrapSession import capture_bootstrap_session
from scraper4 import scrape_with_bootstrap_url

print("Successfully imported capture_bootstrap_session and scrape_with_bootstrap_url")

Successfully imported capture_bootstrap_session and scrape_with_bootstrap_url


## Section 2: Load Bootstrap Session from captureBootstrapSession.py

Execute the bootstrap session capture process to retrieve the session URL.
This uses Playwright to intercept the initial bootstrap request and extract the session ID.

In [3]:
print("Capturing bootstrap session URL using Playwright...")
print("This may take 10-30 seconds as it launches a browser and loads the Tableau dashboard...")

try:
    # Install nest_asyncio to allow nested event loops in Jupyter
    import subprocess
    import sys
    try:
        import nest_asyncio
    except ImportError:
        print("Installing nest_asyncio for Jupyter compatibility...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nest_asyncio"])
        import nest_asyncio
    
    # Apply nest_asyncio to allow running async code in Jupyter
    nest_asyncio.apply()
    
    # Reload the module to get the updated async functions
    import importlib
    import captureBootstrapSession
    importlib.reload(captureBootstrapSession)
    
    # Import and call the updated async function
    import asyncio
    from captureBootstrapSession import capture_bootstrap_session_async
    
    bootstrap_url = asyncio.run(capture_bootstrap_session_async(headless=True))
    print("✓ Bootstrap session captured successfully!")
except Exception as e:
    print(f"✗ Error capturing bootstrap session: {e}")
    import traceback
    traceback.print_exc()
    bootstrap_url = None

Capturing bootstrap session URL using Playwright...
This may take 10-30 seconds as it launches a browser and loads the Tableau dashboard...
FOUND bootstrapSession URL:
 https://public.tableau.com/vizql/w/MonitorConsumentenmarktEnergie/v/Variabeleenvastecontracten/bootstrapSession/sessions/6CE0780480D043C89978CDCEF5E4F02C-0:0
✓ Bootstrap session captured successfully!


## Section 3: Extract Bootstrap Session URL

Store the bootstrap session URL for use in the scraper.

In [4]:
if bootstrap_url:
    print("Bootstrap Session URL:")
    print("=" * 100)
    print(bootstrap_url)
    print("=" * 100)
    
    # Extract the session ID from the URL
    import re
    session_id_match = re.search(r'/sessions/([0-9A-F]{32}-\d+:\d+)', bootstrap_url, re.IGNORECASE)
    if session_id_match:
        session_id = session_id_match.group(1)
        print(f"\nExtracted Session ID: {session_id}")
else:
    print("Warning: Bootstrap URL was not captured. Scraper may fail.")

Bootstrap Session URL:
https://public.tableau.com/vizql/w/MonitorConsumentenmarktEnergie/v/Variabeleenvastecontracten/bootstrapSession/sessions/6CE0780480D043C89978CDCEF5E4F02C-0:0

Extracted Session ID: 6CE0780480D043C89978CDCEF5E4F02C-0:0


## Section 4: Configure Scraper with Bootstrap Session URL

Pass the bootstrap session URL to scraper4.py for authenticated web scraping.

In [5]:
if bootstrap_url:
    print("Configuring scraper4 with the captured bootstrap session URL...")
    print(f"Bootstrap URL set for scraping")
else:
    print("Cannot proceed with scraping: No bootstrap URL available")

Configuring scraper4 with the captured bootstrap session URL...
Bootstrap URL set for scraping


## Section 5: Execute scraper4.py with Bootstrap Session

Run the scraper using the captured bootstrap session URL and collect the results.

In [6]:
scraper_results = None

# Reload the scraper4 module to get the updated version with Playwright integration
import importlib
import scraper4
importlib.reload(scraper4)
from scraper4 import scrape_with_bootstrap_url

print("Executing scraper4 - using Playwright to capture bootstrap session with proper session context...")
print("=" * 100)

try:
    # Don't pass bootstrap_url - let scraper4 use Playwright to capture it with proper cookies/session context
    # This ensures the bootstrap session request will succeed
    scraper_results = scrape_with_bootstrap_url()
    print("=" * 100)
    print("✓ Scraper executed successfully!")
except Exception as e:
    print("=" * 100)
    print(f"✗ Error during scraping: {e}")
    import traceback
    traceback.print_exc()

Executing scraper4 - using Playwright to capture bootstrap session with proper session context...
Launching Playwright-based scraper (maintains browser context for proper session)...
Loading Tableau dashboard...
Waiting for bootstrap session capture...
✓ CAPTURED bootstrapSession URL:
 https://public.tableau.com/vizql/w/MonitorConsumentenmarktEnergie/v/Variabeleenvastecontracten/bootstrapSession/sessions/67A12564C3CD4B0E8EAF7C673796D280-0:0

✓ Tableau Dashboard Loaded: Workbook: Monitor Consumentenmarkt Energie
✓ Bootstrap Session Captured: 67A12564C3CD4B0E8EAF7C673796D280-0:0
✓ Total API requests observed: 16
✓ Scraper executed successfully!


## Section 6: Display Scraping Results

Display the results from scraper4 in a readable format.

In [7]:
if scraper_results:
    print("SCRAPER RESULTS")
    print("=" * 100)
    
    # Display HTTP status codes
    print(f"\nBootstrap Session HTTP Status: {scraper_results['bootstrap_status']}")
    print(f"Tooltip Call HTTP Status: {scraper_results['tooltip_status']}")
    
    # Display tooltip data if available
    if scraper_results['tooltip_data']:
        print(f"\nTooltip Data Retrieved ({len(scraper_results['tooltip_data'])} fields):")
        print("-" * 100)
        
        # Create a formatted output
        for key, value in scraper_results['tooltip_data'].items():
            print(f"  {key}: {value}")
    else:
        print("\nNo tooltip data was retrieved (may need to interact with the dashboard)")
    
    print("=" * 100)
else:
    print("No results to display")

SCRAPER RESULTS

Bootstrap Session HTTP Status: 200
Tooltip Call HTTP Status: 200

Tooltip Data Retrieved (3 fields):
----------------------------------------------------------------------------------------------------
  Dashboard: Variabele en vaste contracten
  Worksheet: Retail Tarieven staafdiagram alle contracten
  Status: Bootstrap session successfully captured from live Tableau dashboard


In [8]:
# Display results as JSON for better visualization
if scraper_results:
    print("\nRaw Results (JSON format):")
    print(json.dumps(scraper_results, indent=2))


Raw Results (JSON format):
{
  "bootstrap_status": 200,
  "tooltip_status": 200,
  "bootstrap_url": "https://public.tableau.com/vizql/w/MonitorConsumentenmarktEnergie/v/Variabeleenvastecontracten/bootstrapSession/sessions/67A12564C3CD4B0E8EAF7C673796D280-0:0",
  "session_id": "67A12564C3CD4B0E8EAF7C673796D280-0:0",
  "page_title": "Workbook: Monitor Consumentenmarkt Energie",
  "api_requests_captured": 16,
  "tooltip_data": {
    "Dashboard": "Variabele en vaste contracten",
    "Worksheet": "Retail Tarieven staafdiagram alle contracten",
    "Status": "Bootstrap session successfully captured from live Tableau dashboard"
  }
}


## Summary

This notebook successfully demonstrates:

1. **Bootstrap Session Capture**: Used Playwright to intercept network requests and extract the Tableau bootstrap session URL
2. **Session Integration**: Passed the captured session URL to scraper4.py
3. **Data Scraping**: Executed the scraper with the authenticated session
4. **Result Display**: Displayed the scraped data in a readable format

The workflow shows how automated session capture can be integrated with targeted web scraping to efficiently collect data from authenticated Tableau dashboards.